In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

ANALYTICS_DIR = PROJECT_DIR / "data" / "analytics_dataset"

district_data = pd.read_csv(
    ANALYTICS_DIR / "KAWASAN_Unified_District_Dataset_2024.csv"
)

state_data = pd.read_csv(
    ANALYTICS_DIR / "KAWASAN_Unified_State_Dataset_2010_2025.csv"
)

gdp_data = pd.read_csv(
    ANALYTICS_DIR / "KAWASAN_GDP_District_2015_2020.csv"
)

print("KAWASAN Analytics Environment Ready!")

print("\nDistrict Dataset:", district_data.shape)
print("State Dataset:", state_data.shape)
print("Historical GDP Dataset:", gdp_data.shape)

print("\nDistrict indicators:")
print(district_data.columns.tolist())

KAWASAN Analytics Environment Ready!

District Dataset: (155, 33)
State Dataset: (256, 12)
Historical GDP Dataset: (10626, 14)

District indicators:
['state_clean', 'district_clean', 'code_state', 'code_district', 'code_state_district', 'income_mean', 'income_median', 'gini', 'poverty_absolute', 'poverty_relative', 'piped_water', 'sanitation', 'electricity', 'population', 'enrolment_primary', 'enrolment_secondary', 'enrolment_post_secondary', 'completion_primary', 'completion_secondary_lower', 'completion_secondary_upper', 'schools_primary', 'schools_secondary', 'schools_tertiary', 'teachers_primary', 'teachers_secondary', 'lf', 'lf_employed', 'lf_unemployed', 'lf_outside', 'p_rate', 'u_rate', 'ep_ratio', 'expenditure_mean']


In [2]:
def get_district_profile(state, district):
    match = district_data[
        (district_data["state_clean"].str.lower() == state.lower()) &
        (district_data["district_clean"].str.lower() == district.lower())
    ]

    if match.empty:
        return {"error": "District not found"}

    row = match.iloc[0]

    def values(columns):
        return {
            col: None if pd.isna(row[col]) else row[col].item()
            if isinstance(row[col], np.generic) else row[col]
            for col in columns
        }

    profile = {
        "geography": {
            "state": row["state_clean"],
            "district": row["district_clean"],
            "district_code": row["code_state_district"],
            "year": 2024
        },

        "economic": values([
            "income_mean",
            "income_median",
            "expenditure_mean",
            "gini",
            "poverty_absolute",
            "poverty_relative"
        ]),

        "education": values([
            "enrolment_primary",
            "enrolment_secondary",
            "enrolment_post_secondary",
            "completion_primary",
            "completion_secondary_lower",
            "completion_secondary_upper",
            "schools_primary",
            "schools_secondary",
            "schools_tertiary",
            "teachers_primary",
            "teachers_secondary"
        ]),

        "employment": values([
            "lf",
            "lf_employed",
            "lf_unemployed",
            "lf_outside",
            "p_rate",
            "u_rate",
            "ep_ratio"
        ]),

        "infrastructure": values([
            "piped_water",
            "sanitation",
            "electricity"
        ]),

        "demographics": values([
            "population"
        ])
    }

    return profile


# Test with Johor Bahru
profile = get_district_profile("johor", "johor bahru")

import json
print(json.dumps(profile, indent=2))

{
  "geography": {
    "state": "johor",
    "district": "johor bahru",
    "district_code": "1_2",
    "year": 2024
  },
  "economic": {
    "income_mean": 10975,
    "income_median": 8977,
    "expenditure_mean": 6764,
    "gini": 0.35,
    "poverty_absolute": 1.9,
    "poverty_relative": 9.9
  },
  "education": {
    "enrolment_primary": 76956.0,
    "enrolment_secondary": 49824.0,
    "enrolment_post_secondary": 1368.0,
    "completion_primary": 95.9,
    "completion_secondary_lower": 99.7,
    "completion_secondary_upper": 97.4,
    "schools_primary": 184.0,
    "schools_secondary": 87.0,
    "schools_tertiary": 1.0,
    "teachers_primary": 5042.0,
    "teachers_secondary": 3899.0
  },
  "employment": {
    "lf": 921.3,
    "lf_employed": 896.3,
    "lf_unemployed": 25.0,
    "lf_outside": 314.1,
    "p_rate": 74.6,
    "u_rate": 2.7,
    "ep_ratio": 72.6
  },
  "infrastructure": {
    "piped_water": 99.99,
    "sanitation": 100.0,
    "electricity": 100.0
  },
  "demographics": {

In [3]:
benchmark_indicators = {
    "income_median": "higher",
    "poverty_absolute": "lower",
    "poverty_relative": "lower",
    "gini": "lower",
    "p_rate": "higher",
    "u_rate": "lower"
}

def benchmark_district(state, district):
    match = district_data[
        (district_data["state_clean"].str.lower() == state.lower()) &
        (district_data["district_clean"].str.lower() == district.lower())
    ]

    if match.empty:
        return {"error": "District not found"}

    row = match.iloc[0]
    results = {}

    for indicator, direction in benchmark_indicators.items():
        values = district_data[indicator].dropna()
        value = row[indicator]

        if pd.isna(value):
            results[indicator] = None
            continue

        if direction == "higher":
            rank = (values > value).sum() + 1
        else:
            rank = (values < value).sum() + 1

        n = len(values)

        results[indicator] = {
            "value": float(value),
            "district_median": float(values.median()),
            "rank": int(rank),
            "out_of": n,
            "percentile": round(
                100 * (n - rank) / (n - 1), 1
            ) if n > 1 else 100.0,
            "direction": direction
        }

    return {
        "state": state,
        "district": district,
        "year": 2024,
        "benchmarks": results
    }


benchmark = benchmark_district("johor", "johor bahru")

print(json.dumps(benchmark, indent=2))

{
  "state": "johor",
  "district": "johor bahru",
  "year": 2024,
  "benchmarks": {
    "income_median": {
      "value": 8977.0,
      "district_median": 4606.0,
      "rank": 6,
      "out_of": 155,
      "percentile": 96.8,
      "direction": "higher"
    },
    "poverty_absolute": {
      "value": 1.9,
      "district_median": 8.0,
      "rank": 14,
      "out_of": 155,
      "percentile": 91.6,
      "direction": "lower"
    },
    "poverty_relative": {
      "value": 9.9,
      "district_median": 14.3,
      "rank": 43,
      "out_of": 155,
      "percentile": 72.7,
      "direction": "lower"
    },
    "gini": {
      "value": 0.35,
      "district_median": 0.328,
      "rank": 112,
      "out_of": 155,
      "percentile": 27.9,
      "direction": "lower"
    },
    "p_rate": {
      "value": 74.6,
      "district_median": 66.8,
      "rank": 11,
      "out_of": 155,
      "percentile": 93.5,
      "direction": "higher"
    },
    "u_rate": {
      "value": 2.7,
      "district

In [4]:
def analyse_development_gaps(state, district):
    benchmark = benchmark_district(state, district)

    if "error" in benchmark:
        return benchmark

    results = {
        "relative_strengths": [],
        "middle_range": [],
        "potential_gaps": []
    }

    for indicator, data in benchmark["benchmarks"].items():

        if data is None:
            continue

        score = data["percentile"]

        item = {
            "indicator": indicator,
            "value": data["value"],
            "district_median": data["district_median"],
            "rank": data["rank"],
            "out_of": data["out_of"],
            "percentile": score
        }

        if score >= 75:
            results["relative_strengths"].append(item)

        elif score >= 25:
            results["middle_range"].append(item)

        else:
            results["potential_gaps"].append(item)

    return {
        "state": state,
        "district": district,
        "year": 2024,
        "analysis": results
    }


gap_analysis = analyse_development_gaps(
    "johor", "johor bahru"
)

print(json.dumps(gap_analysis, indent=2))

{
  "state": "johor",
  "district": "johor bahru",
  "year": 2024,
  "analysis": {
    "relative_strengths": [
      {
        "indicator": "income_median",
        "value": 8977.0,
        "district_median": 4606.0,
        "rank": 6,
        "out_of": 155,
        "percentile": 96.8
      },
      {
        "indicator": "poverty_absolute",
        "value": 1.9,
        "district_median": 8.0,
        "rank": 14,
        "out_of": 155,
        "percentile": 91.6
      },
      {
        "indicator": "p_rate",
        "value": 74.6,
        "district_median": 66.8,
        "rank": 11,
        "out_of": 155,
        "percentile": 93.5
      }
    ],
    "middle_range": [
      {
        "indicator": "poverty_relative",
        "value": 9.9,
        "district_median": 14.3,
        "rank": 43,
        "out_of": 155,
        "percentile": 72.7
      },
      {
        "indicator": "gini",
        "value": 0.35,
        "district_median": 0.328,
        "rank": 112,
        "out_of": 155,


In [5]:
indicator_labels = {
    "income_median": "Median household income",
    "poverty_absolute": "Absolute poverty",
    "poverty_relative": "Relative poverty",
    "gini": "Income inequality",
    "p_rate": "Labour force participation",
    "u_rate": "Unemployment"
}

def generate_district_insights(state, district):
    result = analyse_development_gaps(state, district)

    if "error" in result:
        return result

    analysis = result["analysis"]
    insights = []

    for category, heading in [
        ("relative_strengths", "Relative strength"),
        ("middle_range", "Middle-range indicator"),
        ("potential_gaps", "Potential development gap")
    ]:
        for item in analysis[category]:
            label = indicator_labels.get(
                item["indicator"], item["indicator"]
            )

            insights.append({
                "category": category,
                "indicator": item["indicator"],
                "headline": f"{heading}: {label}",
                "description": (
                    f"{label} ranks {item['rank']} out of "
                    f"{item['out_of']} districts, with a "
                    f"favourable rank position of "
                    f"{item['percentile']}%."
                )
            })

    # Highlight comparatively weak indicators,
    # even if they do not cross the gap threshold.
    weakest = sorted(
        [
            item
            for group in analysis.values()
            for item in group
        ],
        key=lambda x: x["percentile"]
    )

    return {
        "state": state,
        "district": district,
        "year": 2024,
        "insights": insights,
        "lowest_ranked_indicators": [
            indicator_labels.get(
                item["indicator"], item["indicator"]
            )
            for item in weakest[:3]
        ]
    }


insights = generate_district_insights(
    "johor", "johor bahru"
)

print(json.dumps(insights, indent=2))

{
  "state": "johor",
  "district": "johor bahru",
  "year": 2024,
  "insights": [
    {
      "category": "relative_strengths",
      "indicator": "income_median",
      "headline": "Relative strength: Median household income",
      "description": "Median household income ranks 6 out of 155 districts, with a favourable rank position of 96.8%."
    },
    {
      "category": "relative_strengths",
      "indicator": "poverty_absolute",
      "headline": "Relative strength: Absolute poverty",
      "description": "Absolute poverty ranks 14 out of 155 districts, with a favourable rank position of 91.6%."
    },
    {
      "category": "relative_strengths",
      "indicator": "p_rate",
      "headline": "Relative strength: Labour force participation",
      "description": "Labour force participation ranks 11 out of 155 districts, with a favourable rank position of 93.5%."
    },
    {
      "category": "middle_range",
      "indicator": "poverty_relative",
      "headline": "Middle-range 

In [6]:
additional_indicators = [
    "completion_primary",
    "completion_secondary_lower",
    "completion_secondary_upper",
    "piped_water",
    "sanitation",
    "electricity",
    "schools_primary",
    "schools_secondary",
    "teachers_primary",
    "teachers_secondary",
    "population"
]

coverage = pd.DataFrame({
    "available": district_data[additional_indicators].notna().sum(),
    "missing": district_data[additional_indicators].isna().sum(),
    "minimum": district_data[additional_indicators].min(),
    "median": district_data[additional_indicators].median(),
    "maximum": district_data[additional_indicators].max()
})

coverage["coverage_pct"] = (
    coverage["available"] / len(district_data) * 100
).round(1)

print(coverage.to_string())

                            available  missing  minimum   median  maximum  coverage_pct
completion_primary                124       31    90.90    99.55    151.2          80.0
completion_secondary_lower        124       31    84.80    97.05    106.5          80.0
completion_secondary_upper        124       31    91.50    96.95    100.7          80.0
piped_water                       155        0    25.00    98.99    100.0         100.0
sanitation                        155        0    91.29   100.00    100.0         100.0
electricity                       155        0    54.17   100.00    100.0         100.0
schools_primary                   144       11     1.00    46.00    184.0          92.9
schools_secondary                 142       13     1.00    12.50     87.0          91.6
teachers_primary                  124       31   239.00  1209.00   6324.0          80.0
teachers_secondary                124       31    49.00   888.00   4485.0          80.0
population                      

In [7]:
completion_indicators = [
    "completion_primary",
    "completion_secondary_lower",
    "completion_secondary_upper"
]

for indicator in completion_indicators:

    above_100 = district_data[
        district_data[indicator] > 100
    ][
        ["state_clean", "district_clean", indicator]
    ].sort_values(
        by=indicator,
        ascending=False
    )

    print(f"\n{'=' * 60}")
    print(f"INDICATOR: {indicator}")
    print(f"Districts exceeding 100: {len(above_100)}")
    print(f"{'=' * 60}")

    if not above_100.empty:
        print(above_100.to_string(index=False))


INDICATOR: completion_primary
Districts exceeding 100: 55
    state_clean       district_clean  completion_primary
        sarawak               serian               151.2
       selangor         kuala langat               110.9
       selangor       kuala selangor               108.6
          sabah             tambunan               107.9
          sabah               tuaran               107.1
          sabah             semporna               107.0
          kedah              pendang               106.8
     terengganu               marang               106.7
          kedah               baling               106.6
          sabah                kunak               105.8
       kelantan          pasir puteh               105.6
          sabah              beluran               104.6
          sabah          kota marudu               104.4
          sabah          kuala penyu               104.4
          sabah                kudat               104.4
       kelantan              

In [8]:
# Add infrastructure indicators to the benchmarking engine

benchmark_indicators.update({
    "piped_water": "higher",
    "sanitation": "higher",
    "electricity": "higher"
})

indicator_labels.update({
    "piped_water": "Piped water access",
    "sanitation": "Sanitation access",
    "electricity": "Electricity access"
})

# Recalculate benchmarking
benchmark = benchmark_district("johor", "johor bahru")

# Recalculate development gaps
gap_analysis = analyse_development_gaps(
    "johor", "johor bahru"
)

# Regenerate insights
insights = generate_district_insights(
    "johor", "johor bahru"
)

print("Total benchmarking indicators:", len(benchmark_indicators))

print("\nInfrastructure benchmarking:")
for indicator in ["piped_water", "sanitation", "electricity"]:
    print(indicator, benchmark["benchmarks"][indicator])

print("\nUpdated development gap summary:")
for category, items in gap_analysis["analysis"].items():
    print(f"{category}: {len(items)}")

Total benchmarking indicators: 9

Infrastructure benchmarking:
piped_water {'value': 99.99, 'district_median': 98.99, 'rank': 65, 'out_of': 155, 'percentile': np.float64(58.4), 'direction': 'higher'}
sanitation {'value': 100.0, 'district_median': 100.0, 'rank': 1, 'out_of': 155, 'percentile': np.float64(100.0), 'direction': 'higher'}
electricity {'value': 100.0, 'district_median': 100.0, 'rank': 1, 'out_of': 155, 'percentile': np.float64(100.0), 'direction': 'higher'}

Updated development gap summary:
relative_strengths: 5
middle_range: 4
potential_gaps: 0


In [9]:
def analyse_infrastructure_access(state, district):

    profile = get_district_profile(state, district)

    if "error" in profile:
        return profile

    infrastructure = profile["infrastructure"]
    results = {}

    for indicator, value in infrastructure.items():

        if value is None:
            results[indicator] = {
                "access": None,
                "shortfall_pp": None,
                "classification": "Data unavailable"
            }
            continue

        shortfall = max(0, 100 - float(value))

        if shortfall <= 1:
            classification = "Near-universal access"
        elif shortfall <= 5:
            classification = "Minor access gap"
        elif shortfall <= 15:
            classification = "Moderate access gap"
        else:
            classification = "Substantial access gap"

        results[indicator] = {
            "access": float(value),
            "shortfall_pp": round(shortfall, 2),
            "classification": classification
        }

    return {
        "state": state,
        "district": district,
        "year": 2024,
        "infrastructure_access": results
    }


# Test Johor Bahru
jb_access = analyse_infrastructure_access(
    "johor", "johor bahru"
)

# Test a district with the lowest piped water access
lowest_water = district_data.loc[
    district_data["piped_water"].idxmin()
]

low_access = analyse_infrastructure_access(
    lowest_water["state_clean"],
    lowest_water["district_clean"]
)

print("JOHOR BAHRU")
print(json.dumps(jb_access, indent=2))

print("\nDISTRICT WITH LOWEST PIPED WATER ACCESS")
print(json.dumps(low_access, indent=2))

JOHOR BAHRU
{
  "state": "johor",
  "district": "johor bahru",
  "year": 2024,
  "infrastructure_access": {
    "piped_water": {
      "access": 99.99,
      "shortfall_pp": 0.01,
      "classification": "Near-universal access"
    },
    "sanitation": {
      "access": 100.0,
      "shortfall_pp": 0,
      "classification": "Near-universal access"
    },
    "electricity": {
      "access": 100.0,
      "shortfall_pp": 0,
      "classification": "Near-universal access"
    }
  }
}

DISTRICT WITH LOWEST PIPED WATER ACCESS
{
  "state": "sarawak",
  "district": "bukit mabong",
  "year": 2024,
  "infrastructure_access": {
    "piped_water": {
      "access": 25.0,
      "shortfall_pp": 75.0,
      "classification": "Substantial access gap"
    },
    "sanitation": {
      "access": 100.0,
      "shortfall_pp": 0,
      "classification": "Near-universal access"
    },
    "electricity": {
      "access": 54.17,
      "shortfall_pp": 45.83,
      "classification": "Substantial access gap"
 

In [10]:
def generate_development_intelligence(state, district):

    profile = get_district_profile(state, district)

    if "error" in profile:
        return profile

    benchmarks = benchmark_district(state, district)
    gaps = analyse_development_gaps(state, district)
    infrastructure = analyse_infrastructure_access(state, district)
    insights = generate_district_insights(state, district)

    # Identify substantial infrastructure access gaps
    priority_infrastructure = []

    for indicator, data in infrastructure["infrastructure_access"].items():

        if data["shortfall_pp"] is None:
            continue

        if data["shortfall_pp"] > 15:
            priority_infrastructure.append({
                "indicator": indicator,
                "access": data["access"],
                "shortfall_pp": data["shortfall_pp"],
                "classification": data["classification"]
            })

    priority_infrastructure.sort(
        key=lambda x: x["shortfall_pp"],
        reverse=True
    )

    return {
        "geography": profile["geography"],
        "profile": profile,
        "benchmarks": benchmarks["benchmarks"],
        "relative_development_analysis": gaps["analysis"],
        "infrastructure_access": infrastructure["infrastructure_access"],
        "priority_infrastructure_gaps": priority_infrastructure,
        "insights": insights["insights"]
    }


# Test with Bukit Mabong
intelligence = generate_development_intelligence(
    "sarawak", "bukit mabong"
)

print("DISTRICT:", intelligence["geography"]["district"])

print("\nPRIORITY INFRASTRUCTURE GAPS:")
print(json.dumps(
    intelligence["priority_infrastructure_gaps"],
    indent=2
))

print("\nRELATIVE DEVELOPMENT ANALYSIS:")
for category, items in intelligence["relative_development_analysis"].items():
    print(f"{category}: {len(items)}")

DISTRICT: bukit mabong

PRIORITY INFRASTRUCTURE GAPS:
[
  {
    "indicator": "piped_water",
    "access": 25.0,
    "shortfall_pp": 75.0,
    "classification": "Substantial access gap"
  },
  {
    "indicator": "electricity",
    "access": 54.17,
    "shortfall_pp": 45.83,
    "classification": "Substantial access gap"
  }
]

RELATIVE DEVELOPMENT ANALYSIS:
relative_strengths: 1
middle_range: 4
potential_gaps: 4


In [11]:
krdi_indicators = {
    "income_median": "higher",
    "poverty_absolute": "lower",
    "poverty_relative": "lower",
    "gini": "lower",
    "piped_water": "higher",
    "electricity": "higher"
}

krdi_readiness = []

for indicator, direction in krdi_indicators.items():

    values = district_data[indicator]

    krdi_readiness.append({
        "indicator": indicator,
        "direction": direction,
        "available": int(values.notna().sum()),
        "missing": int(values.isna().sum()),
        "minimum": float(values.min()),
        "median": float(values.median()),
        "maximum": float(values.max()),
        "unique_values": int(values.nunique())
    })

readiness_df = pd.DataFrame(krdi_readiness)

print("KRDI INDICATOR READINESS")
print(readiness_df.to_string(index=False))

print("\nDISTRICTS WITH ALL SIX INDICATORS:")
print(
    district_data[list(krdi_indicators)]
    .notna()
    .all(axis=1)
    .sum(),
    "out of",
    len(district_data)
)

KRDI INDICATOR READINESS
       indicator direction  available  missing  minimum   median   maximum  unique_values
   income_median    higher        155        0  2589.00 4606.000 11404.000            153
poverty_absolute     lower        155        0     0.10    8.000    50.000            112
poverty_relative     lower        155        0     0.30   14.300    54.200            117
            gini     lower        155        0     0.16    0.328     0.484             97
     piped_water    higher        155        0    25.00   98.990   100.000             91
     electricity    higher        155        0    54.17  100.000   100.000             21

DISTRICTS WITH ALL SIX INDICATORS:
155 out of 155


In [12]:
def minmax_normalise(series, direction="higher"):

    minimum = series.min()
    maximum = series.max()

    if maximum == minimum:
        return pd.Series(
            np.nan,
            index=series.index
        )

    if direction == "higher":
        scores = (
            (series - minimum) /
            (maximum - minimum)
        )
    else:
        scores = (
            (maximum - series) /
            (maximum - minimum)
        )

    return (scores * 100).round(2)


krdi_data = district_data[
    [
        "state_clean",
        "district_clean",
        "code_state_district"
    ] + list(krdi_indicators.keys())
].copy()

for indicator, direction in krdi_indicators.items():

    krdi_data[f"{indicator}_score"] = (
        minmax_normalise(
            krdi_data[indicator],
            direction
        )
    )

score_columns = [
    f"{indicator}_score"
    for indicator in krdi_indicators
]

print("KRDI NORMALISATION COMPLETE")
print("Shape:", krdi_data.shape)

print("\nNormalised score summary:")
print(
    krdi_data[score_columns]
    .describe()
    .round(2)
    .to_string()
)

print("\nJohor Bahru normalised scores:")

print(
    krdi_data.loc[
        (krdi_data["state_clean"] == "johor") &
        (krdi_data["district_clean"] == "johor bahru"),
        score_columns
    ].to_string(index=False)
)

KRDI NORMALISATION COMPLETE
Shape: (155, 15)

Normalised score summary:
       income_median_score  poverty_absolute_score  poverty_relative_score  gini_score  piped_water_score  electricity_score
count               155.00                  155.00                  155.00      155.00             155.00             155.00
mean                 28.74                   79.37                   69.06       48.29              87.26              97.61
std                  19.81                   17.65                   19.75       13.98              20.42              10.10
min                   0.00                    0.00                    0.00        0.00               0.00               0.00
25%                  15.75                   72.95                   56.03       40.74              84.22             100.00
50%                  22.88                   84.17                   74.03       48.15              98.65             100.00
75%                  37.08                   91.98   

In [13]:
# Step 49: Calculate provisional KRDI

krdi_data["economic_wellbeing_score"] = (
    krdi_data[
        [
            "income_median_score",
            "poverty_absolute_score",
            "poverty_relative_score"
        ]
    ].mean(axis=1)
)

krdi_data["economic_equality_score"] = (
    krdi_data["gini_score"]
)

krdi_data["infrastructure_score"] = (
    krdi_data[
        [
            "piped_water_score",
            "electricity_score"
        ]
    ].mean(axis=1)
)

krdi_data["KRDI"] = (
    0.50 * krdi_data["economic_wellbeing_score"]
    + 0.20 * krdi_data["economic_equality_score"]
    + 0.30 * krdi_data["infrastructure_score"]
).round(2)

# Higher KRDI = more favourable composite score
krdi_data["KRDI_rank"] = (
    krdi_data["KRDI"]
    .rank(method="min", ascending=False)
    .astype(int)
)

print("KRDI CALCULATION COMPLETE")
print("Districts scored:", krdi_data["KRDI"].notna().sum())

print("\nKRDI score distribution:")
print(krdi_data["KRDI"].describe().round(2))

print("\nTOP 10 DISTRICTS:")
print(
    krdi_data.sort_values("KRDI", ascending=False)[
        [
            "state_clean",
            "district_clean",
            "KRDI",
            "KRDI_rank"
        ]
    ].head(10).to_string(index=False)
)

print("\nBOTTOM 10 DISTRICTS:")
print(
    krdi_data.sort_values("KRDI", ascending=True)[
        [
            "state_clean",
            "district_clean",
            "KRDI",
            "KRDI_rank"
        ]
    ].head(10).to_string(index=False)
)

print("\nJOHOR BAHRU:")
print(
    krdi_data.loc[
        (krdi_data["state_clean"] == "johor") &
        (krdi_data["district_clean"] == "johor bahru"),
        [
            "economic_wellbeing_score",
            "economic_equality_score",
            "infrastructure_score",
            "KRDI",
            "KRDI_rank"
        ]
    ].round(2).to_string(index=False)
)

KRDI CALCULATION COMPLETE
Districts scored: 155

KRDI score distribution:
count    155.00
mean      66.92
std       11.58
min       23.92
25%       61.30
50%       68.19
75%       74.85
max       97.79
Name: KRDI, dtype: float64

TOP 10 DISTRICTS:
state_clean district_clean  KRDI  KRDI_rank
   selangor   kuala langat 97.79          1
   selangor    hulu langat 96.36          2
   selangor         sepang 88.62          3
 terengganu    kuala nerus 83.57          4
 terengganu          setiu 82.86          5
      johor          kulai 82.66          6
   selangor          klang 82.65          7
   selangor         gombak 82.49          8
   selangor       petaling 81.09          9
   selangor  hulu selangor 80.73         10

BOTTOM 10 DISTRICTS:
state_clean district_clean  KRDI  KRDI_rank
    sarawak   bukit mabong 23.92        155
      sabah          pitas 25.75        154
      sabah    kota marudu 37.48        153
      sabah          ranau 39.24        152
    sarawak           song

In [14]:
# Step 50: Inspect KRDI component contributions

krdi_data["economic_contribution"] = (
    0.50 * krdi_data["economic_wellbeing_score"]
)

krdi_data["equality_contribution"] = (
    0.20 * krdi_data["economic_equality_score"]
)

krdi_data["infrastructure_contribution"] = (
    0.30 * krdi_data["infrastructure_score"]
)

selected_districts = [
    ("selangor", "kuala langat"),
    ("selangor", "hulu langat"),
    ("johor", "johor bahru"),
    ("sarawak", "bukit mabong")
]

selected = pd.concat([
    krdi_data[
        (krdi_data["state_clean"] == state) &
        (krdi_data["district_clean"] == district)
    ]
    for state, district in selected_districts
])

columns = [
    "state_clean",
    "district_clean",
    "economic_wellbeing_score",
    "economic_equality_score",
    "infrastructure_score",
    "economic_contribution",
    "equality_contribution",
    "infrastructure_contribution",
    "KRDI",
    "KRDI_rank"
]

print("KRDI COMPONENT ANALYSIS")
print(
    selected[columns]
    .round(2)
    .to_string(index=False)
)

print("\nCOMPONENT CORRELATION WITH KRDI")
print(
    krdi_data[
        [
            "economic_wellbeing_score",
            "economic_equality_score",
            "infrastructure_score",
            "KRDI"
        ]
    ]
    .corr()["KRDI"]
    .round(3)
)

KRDI COMPONENT ANALYSIS
state_clean district_clean  economic_wellbeing_score  economic_equality_score  infrastructure_score  economic_contribution  equality_contribution  infrastructure_contribution  KRDI  KRDI_rank
   selangor   kuala langat                     95.59                   100.00                 100.0                  47.79                  20.00                         30.0 97.79          1
   selangor    hulu langat                    100.00                    81.79                 100.0                  50.00                  16.36                         30.0 96.36          2
      johor    johor bahru                     83.68                    41.36                 100.0                  41.84                   8.27                         30.0 80.11         13
    sarawak   bukit mabong                     29.19                    46.60                   0.0                  14.60                   9.32                          0.0 23.92        155

COMPONENT CORRE

In [15]:
# Step 51: KRDI Weight Sensitivity Analysis

weight_scenarios = {
    "baseline": (0.50, 0.20, 0.30),
    "equal_dimensions": (1/3, 1/3, 1/3),
    "economy_focused": (0.60, 0.20, 0.20),
    "infrastructure_focused": (0.30, 0.20, 0.50)
}

components = [
    "economic_wellbeing_score",
    "economic_equality_score",
    "infrastructure_score"
]

for scenario, weights in weight_scenarios.items():

    krdi_data[f"KRDI_{scenario}"] = (
        krdi_data[components]
        .mul(weights)
        .sum(axis=1)
    )

    krdi_data[f"rank_{scenario}"] = (
        krdi_data[f"KRDI_{scenario}"]
        .rank(method="min", ascending=False)
        .astype(int)
    )

# Compare rank changes against baseline
for scenario in weight_scenarios:

    krdi_data[f"rank_change_{scenario}"] = (
        krdi_data["rank_baseline"]
        - krdi_data[f"rank_{scenario}"]
    )

print("KRDI WEIGHT SENSITIVITY ANALYSIS")

for scenario in list(weight_scenarios)[1:]:

    rank_difference = (
        krdi_data[f"rank_change_{scenario}"].abs()
    )

    print(f"\nScenario: {scenario}")
    print(
        "Mean absolute rank change:",
        round(rank_difference.mean(), 2)
    )
    print(
        "Maximum rank change:",
        int(rank_difference.max())
    )
    print(
        "Districts moving 10+ ranks:",
        int((rank_difference >= 10).sum())
    )

print("\nSELECTED DISTRICT RANK COMPARISON")

selected_names = [
    "kuala langat",
    "hulu langat",
    "johor bahru",
    "bukit mabong"
]

rank_columns = [
    "state_clean",
    "district_clean"
] + [
    f"rank_{scenario}"
    for scenario in weight_scenarios
]

print(
    krdi_data[
        krdi_data["district_clean"].isin(selected_names)
    ][rank_columns]
    .sort_values("rank_baseline")
    .to_string(index=False)
)

KRDI WEIGHT SENSITIVITY ANALYSIS

Scenario: equal_dimensions
Mean absolute rank change: 8.25
Maximum rank change: 40
Districts moving 10+ ranks: 53

Scenario: economy_focused
Mean absolute rank change: 3.14
Maximum rank change: 27
Districts moving 10+ ranks: 7

Scenario: infrastructure_focused
Mean absolute rank change: 6.94
Maximum rank change: 46
Districts moving 10+ ranks: 40

SELECTED DISTRICT RANK COMPARISON
state_clean district_clean  rank_baseline  rank_equal_dimensions  rank_economy_focused  rank_infrastructure_focused
   selangor   kuala langat              1                      1                     1                            1
   selangor    hulu langat              2                      2                     2                            2
      johor    johor bahru             13                     30                    12                           16
    sarawak   bukit mabong            155                    155                   154                          155


C:\Users\Muhd Haziq\AppData\Local\Temp\ipykernel_31700\3506562779.py:20: Pandas4Warning: Operation with tuple is deprecated. In a future version these will be treated as scalar-like. To retain the old behavior, explicitly wrap in a Series instead.
  .mul(weights)
C:\Users\Muhd Haziq\AppData\Local\Temp\ipykernel_31700\3506562779.py:20: Pandas4Warning: Operation with tuple is deprecated. In a future version these will be treated as scalar-like. To retain the old behavior, explicitly wrap in a Series instead.
  .mul(weights)
C:\Users\Muhd Haziq\AppData\Local\Temp\ipykernel_31700\3506562779.py:20: Pandas4Warning: Operation with tuple is deprecated. In a future version these will be treated as scalar-like. To retain the old behavior, explicitly wrap in a Series instead.
  .mul(weights)
C:\Users\Muhd Haziq\AppData\Local\Temp\ipykernel_31700\3506562779.py:20: Pandas4Warning: Operation with tuple is deprecated. In a future version these will be treated as scalar-like. To retain the old behavio

In [16]:
# Step 52: KRDI Ranking Stability

# Recalculate scenario scores without the Pandas4Warning
for scenario, weights in weight_scenarios.items():

    weight_series = pd.Series(
        weights,
        index=components
    )

    krdi_data[f"KRDI_{scenario}"] = (
        krdi_data[components]
        .mul(weight_series, axis=1)
        .sum(axis=1)
    )

    krdi_data[f"rank_{scenario}"] = (
        krdi_data[f"KRDI_{scenario}"]
        .rank(method="min", ascending=False)
        .astype(int)
    )

# Calculate ranking consistency
print("SPEARMAN RANK CORRELATION")

for scenario in list(weight_scenarios)[1:]:

    correlation = (
        krdi_data["rank_baseline"]
        .corr(
            krdi_data[f"rank_{scenario}"],
            method="spearman"
        )
    )

    print(f"{scenario}: {correlation:.4f}")

# Identify districts most sensitive to weighting
rank_columns = [
    f"rank_{scenario}"
    for scenario in weight_scenarios
]

krdi_data["rank_range"] = (
    krdi_data[rank_columns].max(axis=1)
    - krdi_data[rank_columns].min(axis=1)
)

print("\nTOP 10 MOST WEIGHT-SENSITIVE DISTRICTS")

print(
    krdi_data.sort_values(
        "rank_range",
        ascending=False
    )[
        [
            "state_clean",
            "district_clean",
            *rank_columns,
            "rank_range"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print("\nRANK STABILITY SUMMARY")

print(
    krdi_data["rank_range"]
    .describe()
    .round(2)
)

SPEARMAN RANK CORRELATION
equal_dimensions: 0.9658
economy_focused: 0.9942
infrastructure_focused: 0.9770

TOP 10 MOST WEIGHT-SENSITIVE DISTRICTS
 state_clean district_clean  rank_baseline  rank_equal_dimensions  rank_economy_focused  rank_infrastructure_focused  rank_range
     sarawak    telang usan             75                     64                    57                          121          64
     sarawak          tatau            109                    121                    82                          139          57
     sarawak         sebauh            104                    104                    85                          129          44
    kelantan   kecil lojing             62                     22                    65                           47          43
       sabah  kota kinabalu             90                    124                    87                          100          37
pulau pinang     timur laut             45                     77               

In [17]:
# Step 53: KRDI Ranking Stability Classification

def classify_rank_stability(rank_range):
    if rank_range <= 5:
        return "Very stable"
    elif rank_range <= 15:
        return "Moderately stable"
    elif rank_range <= 30:
        return "Weight-sensitive"
    else:
        return "Highly weight-sensitive"


krdi_data["rank_stability"] = (
    krdi_data["rank_range"]
    .apply(classify_rank_stability)
)

print("KRDI RANKING STABILITY DISTRIBUTION")

print(
    krdi_data["rank_stability"]
    .value_counts()
    .reindex([
        "Very stable",
        "Moderately stable",
        "Weight-sensitive",
        "Highly weight-sensitive"
    ], fill_value=0)
)

print("\nSELECTED DISTRICT STABILITY")

selected_districts = [
    ("selangor", "kuala langat"),
    ("selangor", "hulu langat"),
    ("johor", "johor bahru"),
    ("sarawak", "bukit mabong"),
    ("sarawak", "telang usan")
]

for state, district in selected_districts:
    row = krdi_data[
        (krdi_data["state_clean"] == state) &
        (krdi_data["district_clean"] == district)
    ]

    if row.empty:
        continue

    print(
        f"{district.title()}, {state.title()}: "
        f"KRDI = {row.iloc[0]['KRDI']:.2f}, "
        f"Baseline rank = {row.iloc[0]['rank_baseline']}, "
        f"Rank range = {row.iloc[0]['rank_range']}, "
        f"Stability = {row.iloc[0]['rank_stability']}"
    )

KRDI RANKING STABILITY DISTRIBUTION
rank_stability
Very stable                37
Moderately stable          64
Weight-sensitive           42
Highly weight-sensitive    12
Name: count, dtype: int64

SELECTED DISTRICT STABILITY
Kuala Langat, Selangor: KRDI = 97.79, Baseline rank = 1, Rank range = 0, Stability = Very stable
Hulu Langat, Selangor: KRDI = 96.36, Baseline rank = 2, Rank range = 0, Stability = Very stable
Johor Bahru, Johor: KRDI = 80.11, Baseline rank = 13, Rank range = 18, Stability = Weight-sensitive
Bukit Mabong, Sarawak: KRDI = 23.92, Baseline rank = 155, Rank range = 1, Stability = Very stable
Telang Usan, Sarawak: KRDI = 68.33, Baseline rank = 75, Rank range = 64, Stability = Highly weight-sensitive


In [18]:
# Step 54: Export KRDI v0.1

EXPORT_DIR = PROJECT_DIR / "data" / "analytics_dataset"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

export_columns = [
    "state_clean",
    "district_clean",
    "code_state_district",

    # Original indicators
    *krdi_indicators.keys(),

    # Normalised indicators
    *score_columns,

    # Dimension scores
    "economic_wellbeing_score",
    "economic_equality_score",
    "infrastructure_score",

    # Baseline KRDI
    "KRDI",
    "KRDI_rank",

    # Dimension contributions
    "economic_contribution",
    "equality_contribution",
    "infrastructure_contribution",

    # Weighting scenarios
    *[f"KRDI_{s}" for s in weight_scenarios],
    *[f"rank_{s}" for s in weight_scenarios],

    # Stability
    "rank_range",
    "rank_stability"
]

krdi_export = krdi_data[export_columns].copy()

# Include methodology metadata
krdi_export["reference_year"] = 2024
krdi_export["index_version"] = "KRDI_v0.1"
krdi_export["methodology"] = (
    "Min-max normalisation; "
    "50% economic wellbeing, "
    "20% economic equality, "
    "30% infrastructure"
)

csv_path = EXPORT_DIR / "KAWASAN_KRDI_District_2024_v0.1.csv"
xlsx_path = EXPORT_DIR / "KAWASAN_KRDI_District_2024_v0.1.xlsx"

krdi_export.to_csv(csv_path, index=False)
krdi_export.to_excel(xlsx_path, index=False)

print("KRDI EXPORT COMPLETE")
print("CSV:", csv_path)
print("Excel:", xlsx_path)
print("Shape:", krdi_export.shape)
print("Missing values:", krdi_export.isna().sum().sum())
print("Unique districts:", krdi_export["code_state_district"].nunique())

KRDI EXPORT COMPLETE
CSV: c:\Users\Muhd Haziq\Desktop\KAWASAN\data\analytics_dataset\KAWASAN_KRDI_District_2024_v0.1.csv
Excel: c:\Users\Muhd Haziq\Desktop\KAWASAN\data\analytics_dataset\KAWASAN_KRDI_District_2024_v0.1.xlsx
Shape: (155, 36)
Missing values: 0
Unique districts: 155


In [19]:
# Step 55: Validate District + KRDI Integration

merged_data = district_data.merge(
    krdi_export[
        [
            "code_state_district",
            "KRDI",
            "KRDI_rank",
            "rank_range",
            "rank_stability"
        ]
    ],
    on="code_state_district",
    how="left",
    validate="one_to_one"
)

print("KAWASAN BACKEND DATA INTEGRATION")

print("Merged dataset shape:", merged_data.shape)

print(
    "Districts with KRDI:",
    merged_data["KRDI"].notna().sum()
)

print(
    "Districts without KRDI:",
    merged_data["KRDI"].isna().sum()
)

print(
    "Duplicate district codes:",
    merged_data["code_state_district"].duplicated().sum()
)

print("\nSAMPLE INTEGRATED RECORD")

sample = merged_data[
    (merged_data["state_clean"] == "johor") &
    (merged_data["district_clean"] == "johor bahru")
]

print(
    sample[
        [
            "state_clean",
            "district_clean",
            "income_median",
            "poverty_absolute",
            "piped_water",
            "electricity",
            "KRDI",
            "KRDI_rank",
            "rank_stability"
        ]
    ].to_string(index=False)
)

KAWASAN BACKEND DATA INTEGRATION
Merged dataset shape: (155, 37)
Districts with KRDI: 155
Districts without KRDI: 0
Duplicate district codes: 0

SAMPLE INTEGRATED RECORD
state_clean district_clean  income_median  poverty_absolute  piped_water  electricity  KRDI  KRDI_rank   rank_stability
      johor    johor bahru           8977               1.9        99.99        100.0 80.11         13 Weight-sensitive


In [20]:
import json
from pathlib import Path

geojson_path = Path("../data/processed/kawasan_district_boundaries.geojson")

with open(geojson_path, "r", encoding="utf-8") as f:
    geojson = json.load(f)

print("GeoJSON type:", geojson.get("type"))
print("Total geographic features:", len(geojson["features"]))

print("\nFirst feature properties:")
print(json.dumps(
    geojson["features"][0]["properties"],
    indent=2,
    ensure_ascii=False
))

print("\nFirst feature geometry type:")
print(geojson["features"][0]["geometry"]["type"])

GeoJSON type: FeatureCollection
Total geographic features: 155

First feature properties:
{
  "state": "Johor",
  "district": "Batu Pahat",
  "code_state": 1,
  "code_district": 1,
  "code_state_district": "1_1"
}

First feature geometry type:
MultiPolygon


In [21]:
import pandas as pd

# Load the KRDI dataset
krdi_path = "../data/analytics_dataset/KAWASAN_KRDI_District_2024_v0.1.csv"

krdi = pd.read_csv(krdi_path)

# Extract district codes
geojson_codes = {
    str(feature["properties"]["code_state_district"])
    for feature in geojson["features"]
}

krdi_codes = set(
    krdi["code_state_district"].astype(str)
)

# Compare identifiers
matched = geojson_codes & krdi_codes
missing_in_geojson = krdi_codes - geojson_codes
missing_in_krdi = geojson_codes - krdi_codes

print("GeoJSON districts:", len(geojson_codes))
print("KRDI districts:", len(krdi_codes))
print("Matching districts:", len(matched))

print("\nMissing in GeoJSON:", len(missing_in_geojson))
print("Missing in KRDI:", len(missing_in_krdi))

print("\nDuplicate GeoJSON codes:",
      len(geojson["features"]) - len(geojson_codes))

print("Duplicate KRDI codes:",
      len(krdi) - len(krdi_codes))

if (
    len(matched) == 155
    and not missing_in_geojson
    and not missing_in_krdi
    and len(geojson["features"]) == len(geojson_codes)
    and len(krdi) == len(krdi_codes)
):
    print("\nSUCCESS: All 155 districts match uniquely!")
else:
    print("\nWARNING: Geographic matching needs investigation.")
    print("Missing in GeoJSON:", sorted(missing_in_geojson))
    print("Missing in KRDI:", sorted(missing_in_krdi))

GeoJSON districts: 155
KRDI districts: 155
Matching districts: 155

Missing in GeoJSON: 0
Missing in KRDI: 0

Duplicate GeoJSON codes: 0
Duplicate KRDI codes: 0

SUCCESS: All 155 districts match uniquely!


In [22]:
import json
from pathlib import Path

# Create a lookup dictionary for KRDI records
krdi_lookup = krdi.set_index(
    "code_state_district"
).to_dict(orient="index")

# Classify districts using exploratory KRDI bands
def classify_krdi(score):
    if score >= 80:
        return "High"
    elif score >= 60:
        return "Moderate"
    elif score >= 40:
        return "Low"
    else:
        return "Very low"

# Build new GeoJSON
merged_geojson = {
    "type": "FeatureCollection",
    "features": []
}

for feature in geojson["features"]:

    properties = feature["properties"].copy()
    district_code = str(properties["code_state_district"])

    krdi_record = krdi_lookup[district_code]

    score = float(krdi_record["KRDI"])

    properties.update({
        "krdi_score": round(score, 2),
        "krdi_rank": int(krdi_record["KRDI_rank"]),
        "rank_stability": krdi_record["rank_stability"],
        "krdi_category": classify_krdi(score),
        "reference_year": 2024,
        "index_version": "KRDI_v0.1"
    })

    merged_geojson["features"].append({
        "type": "Feature",
        "properties": properties,
        "geometry": feature["geometry"]
    })

# Export GeoJSON
output_path = Path(
    "../data/processed/kawasan_krdi_2024_map.geojson"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        merged_geojson,
        f,
        ensure_ascii=False
    )

print("Exported:", output_path)
print("Total map features:", len(merged_geojson["features"]))

print("\nFirst feature properties:")
print(json.dumps(
    merged_geojson["features"][0]["properties"],
    indent=2
))

Exported: ..\data\processed\kawasan_krdi_2024_map.geojson
Total map features: 155

First feature properties:
{
  "state": "Johor",
  "district": "Batu Pahat",
  "code_state": 1,
  "code_district": 1,
  "code_state_district": "1_1",
  "krdi_score": 78.83,
  "krdi_rank": 16,
  "rank_stability": "Moderately stable",
  "krdi_category": "Moderate",
  "reference_year": 2024,
  "index_version": "KRDI_v0.1"
}
